In [3]:
from typing import TypedDict

from langgraph.graph import StateGraph, START, END


class State(TypedDict):
    question: str
    route: str

    tech_answer: str
    billing_answer: str

    final_answer: str

In [7]:
def classify(state: State):
    q = state["question"].lower()

    if "payment" in q or "bill" in q:
        return {"route": "billing"}

    return {"route": "technical"}


def technical_agent(state: State):
    return {"tech_answer": "Restart the application."}


def billing_agent(state: State):
    return {"billing_answer": "Please check your latest invoice."}


def merge(state: State):

    if state["route"] == "technical":
        answer = state["tech_answer"]
    else:
        answer = state["billing_answer"]

    return {"final_answer": answer}


def review(state: State):

    if "human" in state["question"].lower():
        return "escalate"

    return "finish"


def human_agent(state: State):
    print("Escalated to Human")
    return {}



In [ ]:
graph = StateGraph(State)

graph.add_node("classify", classify)
graph.add_node("technical", technical_agent)
graph.add_node("billing", billing_agent)
graph.add_node("merge", merge)
graph.add_node("human", human_agent)

graph.add_edge(START, "classify")

# Parallel execution
graph.add_edge("classify", "technical")
graph.add_edge("classify", "billing")

graph.add_edge("technical", "merge")
graph.add_edge("billing", "merge")


# Conditional Edge
graph.add_conditional_edges(
    "merge",
    review,
    {
        "escalate": "human",
        "finish": END,
    },
)

graph.add_edge("human", END)

app = graph.compile()


In [ ]:
result = app.invoke({"question": "My payment failed"})

print(result)